In [1]:
import os
from pathlib import Path

# Print the current working directory
starting_path = os.getcwd()

# Change directory to sleap-roots
if os.path.basename(os.getcwd()) == "sleap-roots":
    pass
else:
    os.chdir("..")
    
current_path = os.getcwd()

print(f"Starting directory: {starting_path}")
print(f"Changed to directory: {current_path}")


import numpy as np
import pandas as pd
import sleap_roots as sr
from sleap_roots.trait_pipelines import (Pipeline, DicotPipeline, YoungerMonocotPipeline, OlderMonocotPipeline, MultipleDicotPipeline)
from sleap_roots.series import Series
from sleap_roots.summary import get_summary

Starting directory: /Users/andrewoconnor/Projects/sleap-roots/notebooks
Changed to directory: /Users/andrewoconnor/Projects/sleap-roots


---

### DicotPipeline

- Examples
    - `tests/data/canola_7do`
    - `tests/data/soy_6do`

In [85]:
from dataclasses import dataclass

@dataclass
class DicotData:
    # canola_7do
    canola_series_name = "canola"
    canola_id = "919QDUH"
    canola_folder_path = "tests/data/canola_7do"
    canola_h5 = "tests/data/canola_7do/919QDUH.h5"
    canola_primary_slp = "tests/data/canola_7do/919QDUH.primary.predictions.slp"
    canola_lateral_slp = "tests/data/canola_7do/919QDUH.lateral.predictions.slp"
    canola_traits_csv = "tests/919QDUH.traits.csv"
    canola_batch_csv = "tests/919QDUH.batch_traits.csv"

    # soy_6do
    soy_series_name = "soy"
    soy_id = "6PR6AA22JK"
    soy_folder_path = "tests/data/soy_6do"
    soy_h5 = "tests/data/soy_6do"
    soy_primary_slp = "tests/data/soy_6do/6PR6AA22JK.primary.predictions.slp"
    soy_lateral_slp = "tests/data/soy_6do/6PR6AA22JK.lateral.predictions.slp"
    soy_traits_csv = "tests/data/6PR6AA22JK.traits.csv"
    soy_batch_csv = "tests/data/6PR6AA22JK.batch_traits.csv"

    canola = None
    soy = None

    def __post_init__(self):
        self.canola = Series.load(
                        series_name = self.canola_series_name,
                        h5_path = self.canola_h5,
                        primary_path = self.canola_primary_slp,
                        lateral_path= self.canola_lateral_slp,
                    )
        
        self.soy = Series.load(
                        series_name = self.soy_series_name,
                        h5_path = self.soy_h5,
                        primary_path = self.soy_primary_slp,
                        lateral_path= self.soy_lateral_slp,
                    )
        
@dataclass
class MonocotData:
    # rice_3do
    young_rice_path = "tests/data/rice_3do"

    sample_0_h5 = "tests/data/rice_3do/0K9E8BI.h5"
    sample_Y_h5 = "tests/data/rice_3do/YR39SJX.h5"

    sample_0_crown_slp = "tests/data/rice_3do/0K9E8BI.crown.predictions.slp"
    sample_Y_crown_slp = "tests/data/rice_3do/YR39SJX.crown.predictions.slp"

    sample_0_primary_slp = "tests/data/rice_3do/0K9E8BI.primary.predictions.slp"
    sample_Y_primary_slp = "tests/data/rice_3do/YR39SJX.primary.predictions.slp"

    sample_0_traits_csv = "tests/data/rice_3do/0K9E8BI.traits.csv"
    sample_Y_traits_csv = "tests/data/rice_3do/YR39SJX.traits.csv"

    sample_0_batch_traits_csv = "tests/data/rice_3do/0K9E8BI.batch_traits.csv"
    sample_Y_batch_traits_csv = "tests/data/rice_3do/YR39SJX.batch_traits.csv"

    # rice_10do
    old_rice_path =  "tests/data/rice_10do"

    sample_0_crown_slp = "tests/data/rice_10do/0K9E8BI.crown.predictions.slp"
    sample_0_h5 = "tests/data/rice_10do/0K9E8BI.h5"

    younger_rice_YR39SJX = None
    younger_rice_0K9E8BI = None


    def __post_init__(self):
        all_slp_paths = sr.find_all_slp_paths(self.young_rice_path)

        # List of length 2 containing 2 Series objects.
        all_series = sr.load_series_from_slps(slp_paths=all_slp_paths, h5s=True)

        self.younger_rice_YR39SJX = [
            series for series in all_series if series.series_name == "YR39SJX"
        ][0]
        self.younger_rice_0K9E8BI = [
            series for series in all_series if series.series_name == "0K9E8BI"
        ][0]



No, a mixin doesn't need an initialization method (__init__) in Python, as it's primarily a class designed for code reuse and providing functionality, not for creating instances on its own

The order of the mixin matters.

Mixin Methods Are Available First:

Since Python resolves method lookups using C3 linearization (MRO), PipelineMixin methods will be checked first. This ensures that any debugging-related methods are available when DebuggedDicotPipeline is used.

Doesn't Affect Core Behavior:

DicotPipeline (which inherits from Pipeline) retains its normal behavior because it appears later in the inheritance list.

Doesn’t Override Anything:

Even though PipelineMixin does not override any methods, it still makes sense to keep it first. This makes it clear that it’s adding functionality rather than modifying existing behavior.

Best practice: Keep your mixin first in the inheritance list unless there’s a compelling reason not to. Your original order is correct. ✅

In [ ]:
class PipelineMixin:

    def get_initial_traits(self): 

        initial_trait_map = {"DicotPipeline": ["primary_pts", "lateral_pts"],
                             "MultipleDicotPipeline": ["primary_pts", "lateral_pts"],
                             "YoungerMonocotPipeline": ["primary_pts", "crown_pts"],
                             "OlderMonocotPipeline": ["crown_pts"],
                             }

        if isinstance(self, DicotPipeline):
            return initial_trait_map["DicotPipeline"]

        if isinstance(self, YoungerMonocotPipeline):
            return initial_trait_map["YoungerMonocotPipeline"]
        
        if isinstance(self, OlderMonocotPipeline):
            return initial_trait_map["OlderMonocotPipeline"]
        
        else:
            raise NotImplementedError
        
    def get_trait_dict(self: Pipeline, plant: Series, include_summary: bool = True) -> dict[dict]:
        """
        Returns a dictionary where each key is a frame number and the value is the trait dict for that frame.
        Each key of the trait dict is a trait_name and the value is a trait_value
        The trait dict contains every trait possible (including intermediates), representing the entire DAG and more.
        Includes:
            - csv traits
            - summary traits (min, max, mean, median, std, p5, p25, p75, p95)
        """

        num_frames = len(plant)
        trait_dict = {}

        for frame_idx in range(num_frames):

            initial_traits_dict = self.get_initial_frame_traits(plant, frame_idx)

            frame_traits = self.compute_frame_traits(initial_traits_dict)

            if include_summary:
                for trait_name in self.summary_traits:
                    trait_summary = get_summary(frame_traits[trait_name], prefix = f"{trait_name}_")
                    frame_traits.update(trait_summary)
            
            frame_traits["plant_name"] = plant.series_name
            frame_traits["frame_idx"] = frame_idx
            
            trait_dict[frame_idx] = frame_traits

        return trait_dict
    

    # series_name, frame_idx, input_type, input_shape, trait_fn, output_type, output_shape
    def get_traits_and_types_from_plant(self, plant: Series) -> dict:
        """should mostly be a helper..."""
        trait_dict = self.get_trait_dict(plant, include_summary=False)
    
        overall = dict()
        initial_traits = self.get_initial_traits()

        for frame_idx in trait_dict:
            
            frame_traits = trait_dict[frame_idx]
            frame_dict = dict()  # This is where all traits for this frame will be stored
            metadata = ["plant_name", "frame_idx"]

            for trait in frame_traits:
                trait_map = self.trait_map

                if (trait in initial_traits) or (trait in self.summary_traits) or (trait in metadata):
                    continue

                trait_def = trait_map[trait]

                curr_trait_type = type(frame_traits[trait]).__name__

                shape = None
                length = None
                is_nan = False
                has_nan = False
                trait_fn = trait_def.fn
                input_traits = trait_def.input_traits
                kwargs = trait_def.kwargs
                include_in_csv = trait_def.include_in_csv

                if isinstance(frame_traits[trait], np.ndarray):
                    shape = frame_traits[trait].shape
                    if bool(np.all(np.isnan(frame_traits[trait]))):
                        is_nan = True
                    if bool(np.any(np.isnan(frame_traits[trait]))):
                        has_nan = True

                # does not handle the list case

                frame_dict[trait] = {"value": frame_traits[trait],
                                     "output_dtype": curr_trait_type,
                                     "output_shape": shape,
                                     "output_length": length,
                                     "is_nan": is_nan,
                                     "has_nan": has_nan,
                                     "trait_fn": trait_fn.__name__,
                                     "input_traits": input_traits,
                                     "kwargs": kwargs,
                                     "include_in_csv": include_in_csv}

            overall[frame_idx] = frame_dict  # Store all traits for this frame in overall

        return overall
    
    def analyze_trait_dict(self, plant):

        output = self.get_traits_and_types_from_plant(plant)
        series_name = plant.series_name

        # Initialize an empty list to hold the rows
        data_rows = []

        # Loop through the outer dictionary (frame_idx)
        for frame_idx, traits in output.items():
            # Loop through the inner dictionary (trait_name)
            for trait_name, details in traits.items():
                dtype = details.get('output_dtype', 'N/A')
                shape = details.get('output_shape', 'N/A')
                length = details.get('output_length', 'N/A')
                is_nan = details.get('is_nan', 'N/A')
                has_na = details.get('has_nan', 'N/A')
                trait_fn = details.get('trait_fn', 'N/A')
                input_traits = details.get('input_traits', 'N/A')
                kwargs = details.get('kwargs', 'N/A')
                include_in_csv = details.get('include_in_csv', 'N/A')

                # Append a row of data to the list
                data_rows.append({
                    'series_name': series_name,
                    'frame': frame_idx,
                    'include_in_csv': include_in_csv,
                    'trait_fn': trait_fn,
                    'input_traits': input_traits,
                    'kwargs': kwargs,
                    'trait_name': trait_name,
                    'output_dtype': dtype,
                    'output_shape': shape,
                    'output_length': length,
                    'isnan': is_nan,
                    'hasna': has_na
                })

        # Convert the list of rows to a DataFrame
        df = pd.DataFrame(data_rows)

        # Display the DataFrame
        return df
            

class DicotPipelineDebug(PipelineMixin, DicotPipeline):
    pass

class YoungerMonocotPipelineDebug(PipelineMixin, YoungerMonocotPipeline):
    pass

class OlderMonocotPipeline(PipelineMixin, OlderMonocotPipeline):
    pass


In [94]:
r = MonocotData().younger_rice_0K9E8BI

r

Series(series_name='0K9E8BI', h5_path='tests/data/rice_3do/0K9E8BI.h5', primary_path='tests/data/rice_3do/0K9E8BI.primary.predictions.slp', lateral_path=None, crown_path='tests/data/rice_3do/0K9E8BI.crown.predictions.slp', primary_labels=Labels(labeled_frames=72, videos=1, skeletons=1, tracks=0, suggestions=0), lateral_labels=None, crown_labels=Labels(labeled_frames=72, videos=1, skeletons=1, tracks=0, suggestions=0), video=Video(filename="tests/data/rice_3do/0K9E8BI.h5", shape=(72, 1080, 2048, 1), dataset=vol, backend=HDF5Video), csv_path=None)

In [95]:
ym_pipeline_debug = YoungerMonocotPipelineDebug()


a = ym_pipeline_debug.analyze_trait_dict(MonocotData().younger_rice_0K9E8BI)
b = ym_pipeline_debug.analyze_trait_dict(MonocotData().younger_rice_YR39SJX)


In [99]:
b[b['trait_name'] == "pts_all_array"]

,series_name,frame,include_in_csv,trait_fn,input_traits,kwargs,trait_name,output_dtype,output_shape,output_length,isnan,hasna
1,YR39SJX,0,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
35,YR39SJX,1,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(18, 2)",None,False,True
69,YR39SJX,2,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,True
103,YR39SJX,3,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
137,YR39SJX,4,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2279,YR39SJX,67,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
2313,YR39SJX,68,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
2347,YR39SJX,69,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False
2381,YR39SJX,70,False,get_all_pts_array,[crown_pts],{},pts_all_array,ndarray,"(12, 2)",None,False,False


In [71]:
dicot_pipeline_debug = DicotPipelineDebug()


ws = dicot_pipeline_debug.get_trait_dict(DicotData().canola, include_summary=True)
ts = dicot_pipeline_debug.get_trait_dict(DicotData().canola, include_summary=False)